In [6]:
import joblib
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score

print("--- SENTRi-X: (CIC-IDS2017 Enterprise Data) ---")

processed_dir = "../data/processed/"

# 1. Load the locally scaled CIC-IDS2017 data
print("Loading Enterprise Tensors...")
X_cic = joblib.load(os.path.join(processed_dir, "cic_ids2017_X_test.pkl"))
y_cic = joblib.load(os.path.join(processed_dir, "cic_ids2017_y_test.pkl"))

# 2. Split: 20% for "Studying" (Domain Adaptation), 80% for the Final Exam
X_study, X_exam, y_study, y_exam = train_test_split(
    X_cic, y_cic, test_size=0.80, random_state=42, stratify=y_cic
)

print(f"\nData reserved for SENTRi-X to study: {X_study.shape[0]:,} packets")
print(f"Data reserved for the Final Exam: {X_exam.shape[0]:,} packets")

# 3. Domain Adaptation: Transfer Learning (Random Forest Core)
print("\nInitiating Transfer Learning on the Hybrid Engine (Random Forest Core)...")
rf_adapted = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_adapted.fit(X_study, y_study)

# 4. The Final Evaluation (RF)
print("Executing Final Inference with RF...")
adapted_preds = rf_adapted.predict(X_exam)
rf_probs_exam = rf_adapted.predict_proba(X_exam)[:, 1]

print("\n--- SENTRi-X RF Performance on Enterprise IT Data ---")
print(f"RF Adapted Accuracy: {accuracy_score(y_exam, adapted_preds):.4%}")
print("\nRF Detailed Report:")
print(classification_report(y_exam, adapted_preds, target_names=['Normal', 'Attack']))

print("\nConfusion Matrix (RF):")
print(confusion_matrix(y_exam, adapted_preds))
print("\nRF ROC-AUC:", f"{roc_auc_score(y_exam, rf_probs_exam):.4f}")

--- SENTRi-X: (CIC-IDS2017 Enterprise Data) ---
Loading Enterprise Tensors...

Data reserved for SENTRi-X to study: 40,000 packets
Data reserved for the Final Exam: 160,000 packets

Initiating Transfer Learning on the Hybrid Engine (Random Forest Core)...
Executing Final Inference with RF...

--- SENTRi-X RF Performance on Enterprise IT Data ---
RF Adapted Accuracy: 98.1431%

RF Detailed Report:
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99    128399
      Attack       0.94      0.97      0.95     31601

    accuracy                           0.98    160000
   macro avg       0.96      0.98      0.97    160000
weighted avg       0.98      0.98      0.98    160000


Confusion Matrix (RF):
[[126289   2110]
 [   861  30740]]

RF ROC-AUC: 0.9958


In [7]:
# Save the fine-tuned CIC-IDS2017 model to disk
models_dir = "../models/"
save_path_cic = os.path.join(models_dir, "rf_model_cic_ids2017_finetuned.joblib")
joblib.dump(rf_adapted, save_path_cic)
print(f"Success: Adapted model saved to {save_path_cic}")

Success: Adapted model saved to ../models/rf_model_cic_ids2017_finetuned.joblib


In [8]:
#TRANSFER LEARNING FOR CNN & SAVING BOTH MODELS ---
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
import numpy as np

# Ensure models directory is defined for this notebook
models_dir = "../models/"

print("Initiating Transfer Learning on CNN...")

# 1. Load the baseline ToN-IoT CNN model WITHOUT its old optimizer state
cnn_adapted = load_model(os.path.join(models_dir, "cnn_model_ton_iot.h5"), compile=False)

# 2. Recompile with a fresh optimizer for fine-tuning
cnn_adapted.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 3. Reshape tabular data for the CNN
features_count = X_study.shape[1]
X_study_cnn = np.reshape(
    X_study.values if hasattr(X_study, 'values') else X_study,
    (X_study.shape[0], features_count, 1)
)

# Also prepare the exam split for CNN evaluation
X_exam_cnn = np.reshape(
    X_exam.values if hasattr(X_exam, 'values') else X_exam,
    (X_exam.shape[0], features_count, 1)
)

# 4. Fine-tune the CNN on the 20% CIC-IDS2017 study data
cnn_adapted.fit(X_study_cnn, y_study, epochs=3, batch_size=32, validation_split=0.1, verbose=1)

# 5. Evaluate the adapted CNN on the 80% exam data
cnn_probs_exam = cnn_adapted.predict(X_exam_cnn).ravel()
cnn_preds_exam = (cnn_probs_exam >= 0.5).astype(int)

print("\n--- SENTRi-X CNN Performance on Enterprise IT Data ---")
print(f"CNN Adapted Accuracy: {accuracy_score(y_exam, cnn_preds_exam):.4%}")
print("\nCNN Detailed Report:")
print(classification_report(y_exam, cnn_preds_exam, target_names=['Normal', 'Attack']))
print("\nConfusion Matrix (CNN):")
print(confusion_matrix(y_exam, cnn_preds_exam))
print("\nCNN ROC-AUC:", f"{roc_auc_score(y_exam, cnn_probs_exam):.4f}")

# 6. Hybrid Soft-Voting Ensemble (RF + CNN) on exam split
ensemble_probs_exam = (rf_probs_exam + cnn_probs_exam) / 2.0
ensemble_preds_exam = (ensemble_probs_exam >= 0.5).astype(int)

print("\n--- SENTRi-X Hybrid Ensemble Performance (CIC-IDS2017) ---")
print(f"Ensemble Adapted Accuracy: {accuracy_score(y_exam, ensemble_preds_exam):.4%}")
print("\nEnsemble Detailed Report:")
print(classification_report(y_exam, ensemble_preds_exam, target_names=['Normal', 'Attack']))
print("\nEnsemble Confusion Matrix:")
print(confusion_matrix(y_exam, ensemble_preds_exam))
print("\nEnsemble ROC-AUC:", f"{roc_auc_score(y_exam, ensemble_probs_exam):.4f}")

# 7. SAVE BOTH ADAPTED MODELS TO DISK
print("\nSaving fine-tuned models to disk...")

rf_save_path = os.path.join(models_dir, "rf_model_cic_ids2017_finetuned.joblib")
joblib.dump(rf_adapted, rf_save_path)
print(f"Random Forest successfully saved to: {rf_save_path}")

cnn_save_path = os.path.join(models_dir, "cnn_model_cic_ids2017_finetuned.h5")
cnn_adapted.save(cnn_save_path)
print(f"CNN successfully saved to: {cnn_save_path}")

Initiating Transfer Learning on CNN...
Epoch 1/3
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.6906 - loss: 1.7445 - val_accuracy: 0.7912 - val_loss: 0.4576
Epoch 2/3
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7782 - loss: 0.5088 - val_accuracy: 0.8440 - val_loss: 0.4225
Epoch 3/3
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.7930 - loss: 0.4864 - val_accuracy: 0.8468 - val_loss: 0.4126
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step

--- SENTRi-X CNN Performance on Enterprise IT Data ---
CNN Adapted Accuracy: 84.4669%

CNN Detailed Report:
              precision    recall  f1-score   support

      Normal       0.85      0.98      0.91    128399
      Attack       0.82      0.28      0.41     31601

    accuracy                           0.84    160000
   macro avg       0.83      0.63      0.66    160000
weighted avg       0.84      0.84      0.81    160000


Confusion Matrix (CNN):
[[126449   1950]
 [ 22903   8698]]

CNN ROC-AUC: 0.8203

--- SENTRi-

Random Forest successfully saved to: ../models/rf_model_cic_ids2017_finetuned.joblib
CNN successfully saved to: ../models/cnn_model_cic_ids2017_finetuned.h5
